# Labeling Dataset

Builds the image chips to hand-label in Label Studio or Roboflow.

**Targets:** warehouse roofs (polygons), truck trailers (boxes), loading docks (boxes)

Steps:
1. Assign every candidate to a spatial block (its NAIP quarter-quad tile) and hold out ~20% of blocks for **validation**. Splitting by location, not at random, keeps neighbouring chips out of both sets.
2. Sample ~400 chips: mostly candidate warehouses (weighted toward the 60 cm years, where trailers are visible) plus **negative** chips with no large building.
3. Cut 512 × 512 px RGB chips from the NAIP tiles, save them as PNG, and save an index with each chip's map position so labels can be converted back to real coordinates.

Input: `data/interim/warehouse_candidates.gpkg` (from `notebooks/02_warehouse_candidates.ipynb`)

### Libraries

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from PIL import Image
from rasterio.merge import merge
from shapely.geometry import box

# Repo root: the first folder above this notebook that contains data/data_sources.txt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "data_sources.txt").exists())
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
CHIP_DIR = INTERIM / "chips"
NAIP_DIR = Path("/Volumes/Storage/naip")  # external USB drive
UTM = "EPSG:26918"                        # NAIP tiles are in UTM zone 18N (NAD83), metres

SEED = 42
CHIP_PX = 512                 # chip size in pixels (~307 m at 60 cm, 512 m at 1 m)
VAL_SHARE = 0.2               # share of candidates whose blocks go to validation
YEAR_QUOTA = {2022: 150, 2019: 110, 2015: 40, 2010: 40}   # candidate chips per year
HINT_SHARE = 0.7              # share of candidate chips with warehouse_hint
N_NEGATIVE = 60               # chips with no candidate building (2019 and 2022)
MAX_JITTER = 0.25             # shift chip centre by up to 25% of its width, so buildings aren't always centred

rng = np.random.default_rng(SEED)

### Load candidates and NAIP tiles

In [ ]:
assert NAIP_DIR.exists(), f"{NAIP_DIR} not found; plug in the USB drive"

candidates = gpd.read_file(INTERIM / "warehouse_candidates.gpkg", layer="candidates").to_crs(UTM)

naip = gpd.read_file(NAIP_DIR / "naip_index.geojson")
naip = naip[naip["path"].map(lambda p: (NAIP_DIR / p).exists())].to_crs(UTM)
naip["block"] = naip["id"].str.extract(r"m_(\d{7}_[a-z]{2})", expand=False)  # quarter-quad, same in every year
GSD = naip.groupby("year")["gsd"].first().to_dict()

print(f"{len(candidates):,} candidates; NAIP years {GSD}")

### 1. Spatial train / validation split
Each candidate's block is the NAIP quarter-quad (~6 × 7 km) containing its centre. Whole blocks are assigned to validation until they hold ~20% of candidates.

In [ ]:
tiles_2022 = naip.loc[naip["year"] == 2022, ["block", "geometry"]]
centres = gpd.GeoDataFrame(candidates[["cand_id"]], geometry=candidates.centroid, crs=UTM)
block_of = gpd.sjoin(centres, tiles_2022, predicate="within").drop_duplicates("cand_id").set_index("cand_id")["block"]
candidates["block"] = candidates["cand_id"].map(block_of)

block_sizes = candidates["block"].value_counts()
val_blocks, n_val = [], 0
for block in rng.permutation(block_sizes.index):
    if n_val >= VAL_SHARE * len(candidates):
        break
    val_blocks.append(block)
    n_val += block_sizes[block]

candidates["split"] = np.where(candidates["block"].isin(val_blocks), "val", "train")
print(f"Validation blocks: {len(val_blocks)} of {len(block_sizes)}")
pd.crosstab(candidates["county"], candidates["split"], normalize="index").round(2)

### 2. Sample chips
Candidate chips: each building is used once, in one year. Negative chips: random points in the study area whose chip doesn't touch any candidate building.

In [ ]:
def jittered_box(x, y, year):
    """Chip footprint around (x, y), shifted randomly so the building isn't always centred."""
    size = CHIP_PX * GSD[year]
    dx, dy = rng.uniform(-MAX_JITTER, MAX_JITTER, 2) * size
    cx, cy = x + dx, y + dy
    return box(cx - size / 2, cy - size / 2, cx + size / 2, cy + size / 2)


rows, pool = [], candidates.copy()
for year, n in YEAR_QUOTA.items():
    n_hint = round(n * HINT_SHARE)
    picked = pd.concat([
        pool[pool["warehouse_hint"]].sample(n_hint, random_state=rng.integers(1e9)),
        pool[~pool["warehouse_hint"]].sample(n - n_hint, random_state=rng.integers(1e9)),
    ])
    pool = pool.drop(picked.index)
    for c in picked.itertuples():
        rows.append({"chip_id": f"{year}_{c.cand_id}", "kind": "candidate", "cand_id": c.cand_id, "year": year,
                     "warehouse_hint": c.warehouse_hint, "county": c.county, "block": c.block, "split": c.split,
                     "geometry": jittered_box(c.geometry.centroid.x, c.geometry.centroid.y, year)})

study_area = gpd.read_file(RAW / "core_counties.geojson").to_crs(UTM).union_all()
cand_index = candidates.sindex
minx, miny, maxx, maxy = study_area.bounds
n_neg = 0
while n_neg < N_NEGATIVE:
    x, y = rng.uniform(minx, maxx), rng.uniform(miny, maxy)
    year = 2022 if n_neg % 2 == 0 else 2019
    footprint = jittered_box(x, y, year)
    if not study_area.contains(footprint.centroid) or len(cand_index.query(footprint, predicate="intersects")):
        continue
    block = tiles_2022[tiles_2022.contains(footprint.centroid)]["block"]
    if block.empty:
        continue
    rows.append({"chip_id": f"{year}_neg{n_neg:03d}", "kind": "negative", "cand_id": None, "year": year,
                 "warehouse_hint": False, "county": None, "block": block.iloc[0],
                 "split": "val" if block.iloc[0] in val_blocks else "train", "geometry": footprint})
    n_neg += 1

chips = gpd.GeoDataFrame(rows, crs=UTM)
print(f"Sampled {len(chips)} chips")

### Remove overlap between splits
A training chip that overlaps a validation chip (in any year) would leak the same place into both sets, so it's dropped.

In [ ]:
val_chips = chips[chips["split"] == "val"]
overlaps = gpd.sjoin(chips[chips["split"] == "train"], val_chips[["geometry"]], predicate="intersects").index.unique()
chips = chips.drop(overlaps).reset_index(drop=True)
print(f"Dropped {len(overlaps)} training chips that overlapped validation chips")

pd.crosstab([chips["split"], chips["kind"]], chips["year"], margins=True)

### 3. Export chips
Each chip is cut from every NAIP tile it touches (merged if it crosses a tile edge), saved as an RGB PNG under `chips/train/` or `chips/val/`, and recorded in `chips_index.csv` with the affine transform (`a`–`f`) that maps pixel (col, row) to UTM coordinates: `x = a·col + b·row + c`, `y = d·col + e·row + f`.

In [ ]:
def export_chip(chip):
    tiles = naip[(naip["year"] == chip.year) & naip.intersects(chip.geometry)]["path"]
    sources = [rasterio.open(NAIP_DIR / p) for p in tiles]
    try:
        rgb, transform = merge(sources, bounds=chip.geometry.bounds, res=GSD[chip.year], indexes=[1, 2, 3], nodata=0)
    finally:
        for src in sources:
            src.close()
    rgb = rgb[:, :CHIP_PX, :CHIP_PX]  # guard against a 1-px rounding overshoot
    out = CHIP_DIR / chip.split / f"{chip.chip_id}.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(np.moveaxis(rgb, 0, -1)).save(out)
    return {"file": str(out.relative_to(CHIP_DIR)), "width": rgb.shape[2], "height": rgb.shape[1],
            "n_tiles": len(sources), **dict(zip("abcdef", tuple(transform)[:6]))}


records = []
for n, chip in enumerate(chips.itertuples(), 1):
    records.append(export_chip(chip))
    if n % 50 == 0 or n == len(chips):
        print(f"{n}/{len(chips)} chips exported")

index = pd.concat([chips.drop(columns="geometry"), pd.DataFrame(records)], axis=1)
index["gsd"] = index["year"].map(GSD)
index.to_csv(CHIP_DIR / "chips_index.csv", index=False)
chips.to_file(INTERIM / "chips.gpkg", layer="chips", driver="GPKG")
print(f"Saved index to {(CHIP_DIR / 'chips_index.csv').relative_to(ROOT)} and footprints to data/interim/chips.gpkg")
assert (index[["width", "height"]] == CHIP_PX).all().all(), "some chips are not 512 x 512"

### Preview
A random sample of exported chips.

In [ ]:
sample = index.sample(8, random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, row in zip(axes.flat, sample.itertuples()):
    ax.imshow(Image.open(CHIP_DIR / row.file))
    ax.set_title(f"{row.chip_id} · {row.split}", fontsize=9)
    ax.set_axis_off()
plt.tight_layout()